In [1]:
%load_ext autoreload
%autoreload 2

The code in this notebook uses that in ground_effect_design_study/process_ground_effect_case.py.

to then generate the similarity scores (if using a lot of data and not select cases) use ground_effect_design_study/process_ground_effect_w_sim_scores.py

To use this notebook first run that script to process the cases that you want to make. Then import the data into this notebook for exploration.

In [ ]:
import multiprocessing
import os
from itertools import product, repeat

import numpy as np
import pandas as pd
import yaml
from tqdm import tqdm

from design_similarity.similarity_scores import (
    _calc_mp_emd, _calc_mp_similarity)

# Import the data

In [ ]:
# format is MM_DD_YYYY
DATE_TO_IMPORT = "01_29_2026"

In [4]:
base_dir = os.path.join("/mnt/d/ground_effect_processed_data_python/processed_predictions", f"wig_predictions_{DATE_TO_IMPORT}")
assert os.path.exists(base_dir), "PATH DOES NOT EXIST. CHECK DATE."

In [5]:
clustering_data = pd.read_parquet(os.path.join(base_dir, "data_to_cluster_w_feature_distances.parquet"))
normal_clustering_results = pd.read_parquet(os.path.join(base_dir, "normal_all_clusters.parquet"))
offset_clustering_results = pd.read_parquet(os.path.join(base_dir, "offset_all_clusters.parquet"))

In [6]:
clustering_data['normal_cluster_labels'] = normal_clustering_results
clustering_data['offset_cluster_labels'] = offset_clustering_results

In [7]:
similarity_score_path = os.path.join(base_dir, "similarity_scores.parquet")

similarity_scores_loaded = False
if os.path.exists(similarity_score_path):
    print("similarity scores loaded from pre calculation")
    similarity_scores = pd.read_parquet(similarity_score_path)
    similarity_scores_loaded = True

    clustering_data[similarity_scores.columns] = similarity_scores
else:
    print("similarity scores not calculated")



similarity scores loaded from pre calculation


In [8]:
CASE_IDS_TO_DROP = [f"{ii}0{bb}" for ii, bb in product([2,4,6,8], [10,20,30,40])]
CASE_IDS_TO_DROP.extend([f"0{ii}{bb}" for ii, bb in product([1,2,3,4,6,8], [10,20,30,40])])

In [9]:
case_names = clustering_data['case_id'].unique()
case_id_to_naca = {case_id : naca for case_id, naca in zip(case_names, pd.Series(case_names).str.split("_").str[1].tolist())}
clustering_data['naca'] = clustering_data['case_id'].map(case_id_to_naca)
clustering_data = clustering_data[~clustering_data['naca'].isin(CASE_IDS_TO_DROP)]

# Calculate the similarity scores

In [ ]:
# Loading the YAML file
with open(os.path.join(base_dir, "reference_case.yaml"), 'r') as file:
    config = yaml.safe_load(file)

FOCUS_CASE_ID = config['FOCUS_CASE']

# Check offset clusters to see if they seem right.

#### regular similarity score

In [ ]:
if not similarity_scores_loaded:

    for case_count, compare_case in enumerate(clustering_data['case_id'].unique()):

        print(compare_case)
        compare_and_focus = clustering_data[clustering_data['case_id'].isin([FOCUS_CASE_ID, compare_case])].copy()
        compare_and_focus = compare_and_focus.drop(columns='similarity_score', errors='ignore')

        grouped_data = list(compare_and_focus.groupby("coarse_mesh_parent_cell_id"))
        
        with multiprocessing.pool.Pool(12) as pool: # uses all your processors
            one_case_results = pool.starmap(_calc_mp_similarity, zip(grouped_data, repeat("normal_cluster_labels")))

        one_case_results = pd.DataFrame(np.array(one_case_results), columns=['coarse_mesh_parent_cell_id', 'similarity_score'])

        compare_and_focus = pd.merge(
                compare_and_focus.reset_index(drop=False),
                one_case_results,
                on="coarse_mesh_parent_cell_id"
            ).set_index('index')

        clustering_data.loc[compare_and_focus.index, 'similarity_score'] = compare_and_focus['similarity_score'].values


#### Offset similarity score

In [ ]:
if not similarity_scores_loaded:
    for case_count, compare_case in enumerate(clustering_data['case_id'].unique()):

        print(compare_case)
        compare_and_focus = clustering_data[clustering_data['case_id'].isin([FOCUS_CASE_ID, compare_case])].copy()
        compare_and_focus = compare_and_focus.drop(columns='offset_similarity_score', errors='ignore')

        grouped_data = list(compare_and_focus.groupby("coarse_mesh_parent_cell_id"))
        
        with multiprocessing.pool.Pool(12) as pool: # uses all your processors
            one_case_results = pool.starmap(_calc_mp_similarity, zip(grouped_data, repeat("offset_cluster_labels")))

        one_case_results = pd.DataFrame(np.array(one_case_results), columns=['coarse_mesh_parent_cell_id', 'offset_similarity_score'])

        compare_and_focus = pd.merge(
                compare_and_focus.reset_index(drop=False),
                one_case_results,
                on="coarse_mesh_parent_cell_id"
            ).set_index('index')

        clustering_data.loc[compare_and_focus.index, 'offset_similarity_score'] = compare_and_focus['offset_similarity_score'].values

    clustering_data['adjusted_offset_similarity_score'] = \
        clustering_data['offset_similarity_score'] - clustering_data['offset_distance_adjustment']


#### Try thresholding the similarity score

In [ ]:
if not similarity_scores_loaded:

    focus_data = clustering_data[clustering_data['case_id']==FOCUS_CASE_ID].copy()

    offset_distance_column_names = [col for col in clustering_data if "offset_distance" in col][:-1]

    feature_threshold = 0.05 # 5% of the magnitude

    for compare_case_id in clustering_data['case_id'].unique():
        print(compare_case_id)
        if compare_case_id == FOCUS_CASE_ID:
            continue

        
        compare_data = clustering_data[clustering_data['case_id']==compare_case_id]

        both_features_below_threshold = (compare_data[offset_distance_column_names].abs() < feature_threshold).all(axis=1)

        clustering_data.loc[compare_data.index, 'threshold_similarity_score'] = np.where(
                    both_features_below_threshold & (compare_data['similarity_score'] < 0.8), 
                    1, 
                    compare_data['similarity_score'],
                    )
        clustering_data.loc[compare_data.index, 'threshold_adjusted_offset_similarity_score'] = np.where(
                    both_features_below_threshold & (compare_data['adjusted_offset_similarity_score'] < 0.8), 
                    1, 
                    compare_data['adjusted_offset_similarity_score'],
                    )

    


#### Add the shape scores

In [ ]:
from scipy.stats import wasserstein_distance, wasserstein_distance_nd

def calculate_emd(data1, data2, var: list[str] = None):
  
    if var:
        return [wasserstein_distance(data1[var_name], data2[var_name]) for var_name in var]
    else:
        return wasserstein_distance(data1, data2)


def calculate_offset_distance_two_data_sources(data1, data2, var: str = None):
    if var:
        offset_distance = np.median(data2[var],axis=0) - np.median(data1[var],axis=0)
    else:
        offset_distance = np.median(data2,axis=0) - np.median(data1,axis=0)

    return offset_distance



def calculate_shape_distance(data1, data2, var: str = None):

    if (data1.shape[0] <= 5) | (data2.shape[0] <= 5):
        return [0 for var_name in var]

    offset_distance = calculate_offset_distance_two_data_sources(data1, data2, var=var)

    if var:
        data1_var = data1[var]
        data2_adjusted = data2[var] - offset_distance
        return [wasserstein_distance(data1_var[var_name], data2_adjusted[var_name]) for var_name in var]
    
    data1_var = data1
    data2_adjusted = data2 - offset_distance
    if len(data2_adjusted.shape) > 1:
        return [wasserstein_distance(data1_var[:, ii], data2_adjusted[:, ii]) for ii in range(0, data1_var.shape[1])]
    
    return wasserstein_distance(data1_var, data2_adjusted)




In [ ]:
if not similarity_scores_loaded:
    focus_data = clustering_data[clustering_data['case_id']==FOCUS_CASE_ID].copy()

    feature_names = ['U_0_norm', 'U_1_norm']

    for compare_case_id in tqdm(clustering_data['case_id'].unique()):
        print(compare_case_id)
        if compare_case_id == FOCUS_CASE_ID:
            continue

        compare_data = clustering_data[clustering_data['case_id']==compare_case_id].copy()

        for cell_set_id in focus_data['coarse_mesh_parent_cell_id'].unique():

            focus_cell_set_data = focus_data[focus_data['coarse_mesh_parent_cell_id']==cell_set_id]
            compare_cell_set_data = compare_data[compare_data['coarse_mesh_parent_cell_id']==cell_set_id]

            if compare_cell_set_data.empty:
                continue

            compare_data.loc[compare_cell_set_data.index, 'emd'] = np.sum(calculate_emd(focus_cell_set_data, compare_cell_set_data, feature_names))

        clustering_data.loc[compare_data.index, 'emd'] = compare_data['emd']
        
    clustering_data['similarity_emd'] = 1 - clustering_data['emd']

In [ ]:
if not similarity_scores_loaded:
    feature_names = ['U_0_norm', 'U_1_norm']

    for case_count, compare_case in enumerate(clustering_data['case_id'].unique()):

        print(compare_case)
        compare_and_focus = clustering_data[clustering_data['case_id'].isin([FOCUS_CASE_ID, compare_case])].copy()
        compare_and_focus = compare_and_focus.drop(columns='similarity_emd', errors='ignore')
        compare_and_focus = compare_and_focus[['case_id', 'cell_id', 'coarse_mesh_parent_cell_id'] + feature_names]

        grouped_data = list(compare_and_focus.groupby("coarse_mesh_parent_cell_id"))
        
        with multiprocessing.pool.Pool(12) as pool: # uses all your processors
            one_case_results = pool.starmap(_calc_mp_emd, zip(grouped_data, repeat(feature_names), repeat(FOCUS_CASE_ID)))

        one_case_results = pd.DataFrame(np.array(one_case_results), columns=['coarse_mesh_parent_cell_id', 'similarity_emd'])

        compare_and_focus = pd.merge(
                compare_and_focus.reset_index(drop=False),
                one_case_results,
                on="coarse_mesh_parent_cell_id"
            ).set_index('index')

        clustering_data.loc[compare_and_focus.index, 'similarity_emd'] = compare_and_focus['similarity_emd'].values


# Global Similarity Scores

In [ ]:
def calculate_global_similarity(data: pd.DataFrame, similarity_score_name: str):
    """calculates the global similarity. needs to trim out the far field

    Args:
        data (pd.DataFrame): all case data
    """

    cell_set_similarity_mean = data.reset_index().groupby(by=['coarse_mesh_parent_cell_id'])[similarity_score_name].mean()
    cell_set_similarity_std = data.reset_index().groupby(by=['coarse_mesh_parent_cell_id'])[similarity_score_name].std()

    cell_sets_to_include = cell_set_similarity_std.index[((cell_set_similarity_std > 0.03) & (cell_set_similarity_mean < 0.99))]

    trimmed_data_to_cluster = data[data['coarse_mesh_parent_cell_id'].isin(cell_sets_to_include)]

    sim_scores_step_1 = trimmed_data_to_cluster.groupby(by=['case_id', 'coarse_mesh_parent_cell_id'])[similarity_score_name].mean()
    sim_scores_step_2 = sim_scores_step_1.reset_index(drop=False)
    return sim_scores_step_2.groupby(by=['case_id',])[similarity_score_name].mean()


In [ ]:
adjusted_global_sim_score = calculate_global_similarity(
    clustering_data[['case_id', 'coarse_mesh_parent_cell_id', 'threshold_adjusted_offset_similarity_score']], 'threshold_adjusted_offset_similarity_score')

adjusted_global_sim_score = adjusted_global_sim_score.rename("adjusted_global_similarity_score")

adjusted_global_sim_score = adjusted_global_sim_score.reset_index()